<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/09_under_the_hood.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 09 · Under the hood

Eight lessons in, you have built agents that plan, remember, delegate, retry, redact, and pause
for approval. This lesson builds nothing new. It opens the box.

The claim to test: **`create_deep_agent` is `create_agent` plus a middleware stack.** Not a
different framework, not a different runtime — a well-chosen default configuration of things
you could assemble yourself.

**In this lesson:** proving that equivalence, then the LangGraph machinery underneath both.

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.

In [ ]:
# --- snippet:setup v1 ---
%pip install -qq --progress-bar off \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0" \
  "git+https://github.com/langchain-samples/langsmith-studio-nb.git"

import os

try:
    from google.colab import userdata

    key = userdata.get("LANGSMITH_API_KEY")
except Exception:  # not on Colab, or secret unavailable
    from getpass import getpass

    key = os.environ.get("LANGSMITH_API_KEY") or getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_API_KEY"] = key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-09-under-the-hood"

# One constant, used everywhere. Models are served by the LangSmith gateway,
# so this key is the only credential the notebook needs.
MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

---

## 1. What is actually in there

Every middleware in a Deep Agent is inspectable. Build one and look.

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=MODEL,
    system_prompt="You are a helpful assistant.",
)

# Middleware are attached to the compiled graph's nodes. The names tell the story.
nodes = list(agent.get_graph().nodes)
print("graph nodes:")
for n in nodes:
    print(f"  {n}")

In [ ]:
# A more direct look: which middleware classes did the harness assemble?
# Build agents with different arguments and compare their node lists.
variants = {
    "bare":                {},
    "+ subagents":         {"subagents": [{
                               "name": "helper",
                               "description": "Helps with a subtask.",
                               "system_prompt": "You help.",
                           }]},
    "+ interrupt_on":      {"interrupt_on": {"write_file": True}},
}

for label, kwargs in variants.items():
    a = create_deep_agent(model=MODEL, system_prompt="x", **kwargs)
    print(f"{label:16} -> {sorted(a.get_graph().nodes)}")

Notice the nodes change based on **arguments you passed**, not on which class you called. That
is the first hint: `create_deep_agent` is a factory that assembles a configuration.

---

## 2. Building the same thing by hand

Now the actual claim. Here is a plain `create_agent` — the standard LangChain agent constructor
— assembled with the same pieces.

In [ ]:
from deepagents.backends import StateBackend
from deepagents.middleware.filesystem import FilesystemMiddleware
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, TodoListMiddleware

hand_built = create_agent(
    model=MODEL,
    system_prompt="You are a helpful assistant.",
    middleware=[
        TodoListMiddleware(),                          # planning
        FilesystemMiddleware(backend=StateBackend()),  # ls / read_file / write_file / ...
        SummarizationMiddleware(model=MODEL, trigger=("tokens", 8000)),
    ],
)

result = hand_built.invoke({"messages": [{"role": "user", "content":
    "Plan a three-step approach to learning Rust, then save it to plan.md"
}]})

print(result["messages"][-1].text[:400])
print("\nfiles:", list(result.get("files", {})))
print("todos:", len(result.get("todos", [])))

Planning, a filesystem, files in the returned state — the same behaviours you got from
`create_deep_agent` in lesson 01, assembled from parts.

`create_deep_agent` does considerably more than this by default (subagents, prompt caching,
retries on tool-call patches, a carefully ordered stack). But there is no hidden runtime. It is
**configuration, not magic** — which means anything the harness does, you can change.

---

## 3. One layer further down: LangGraph

Both `create_agent` and `create_deep_agent` return a **compiled LangGraph graph**. That is the
runtime underneath everything in this course.

In [ ]:
print(agent.get_graph().draw_mermaid())

Paste that into any Mermaid renderer to see the loop: **model → tools → model → tools → ...**
until the model stops asking for tools.

LangGraph in four nouns:

| Noun | What it is | In your agent |
|---|---|---|
| **State** | a typed dict passed between steps | `DeepAgentState`, extended by middleware |
| **Node** | a function that reads state and returns an update | the model call; the tool executor |
| **Edge** | which node runs next | "did the model request a tool?" |
| **Reducer** | how updates merge into state | `messages` appends rather than overwrites |

Two things worth noticing.

**Middleware extend the state.** `files` and `todos` are not declared on `DeepAgentState` — the
filesystem and planning middleware each contribute their own channels, merged in when the agent
is built. Add a middleware, get new state.

**Reducers decide what an update means.** When a node returns `{"messages": [new_message]}`, the
message is *appended*, because the `messages` channel has an append reducer. State updates are
declarative; the reducer defines merging.

In [ ]:
# The base state schema is an ordinary typed dict. Nothing is hidden.
from deepagents import DeepAgentState

print("declared on DeepAgentState itself:")
for name in DeepAgentState.__annotations__:
    print(f"  {name}")

# Note what is NOT there: `files` and `todos`. Middleware contribute their own state
# channels, which are merged in at build time — so the real schema depends on the stack.
result = agent.invoke({"messages": [{"role": "user", "content":
    "Make a two-item plan for tidying a garage, then save it to plan.md"
}]})
print("\nkeys actually present in a returned state:")
for key in result:
    print(f"  {key}")

---

## 4. Checkpointers, threads, and time travel

A checkpointer saves state after every step. That single mechanism is what gives you
conversation memory, human-in-the-loop, and crash recovery — they are not three features, they
are one.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memoried = create_deep_agent(
    model=MODEL,
    system_prompt="You are a helpful assistant.",
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "demo-thread"}}

memoried.invoke({"messages": [{"role": "user", "content": "My favourite colour is teal."}]}, config=config)
answer = memoried.invoke({"messages": [{"role": "user", "content": "What is my favourite colour?"}]}, config=config)
print(answer["messages"][-1].text)

No memory middleware, no store. The second call remembered because the checkpointer **replayed
the thread's saved state** before running. This is lesson 07's tier zero: conversation memory is
just persistence.

In [ ]:
# Every step was saved, so you can walk backwards through the thread's history.
history = list(memoried.get_state_history(config))
print(f"{len(history)} checkpoints in this thread\n")

for snapshot in history[:6]:
    last = snapshot.values.get("messages", [])
    preview = last[-1].text[:60].replace("\n", " ") if last else "(empty)"
    print(f"  step -{history.index(snapshot):<2} next={str(snapshot.next):<12} {preview}")

In [ ]:
# Time travel: resume from an earlier checkpoint and take a different path.
earlier = history[2]

branched = memoried.invoke(
    {"messages": [{"role": "user", "content": "Actually, what is the capital of France?"}]},
    config=earlier.config,   # <- rewind to this point, then continue
)
print(branched["messages"][-1].text[:200])

You just forked a conversation from the middle. Same mechanism as an undo button, an A/B test
of two different replies, or a debugger that re-runs a failing step with one input changed.

---

## 5. When to drop a layer

Now that you can see the layers, the honest guidance on choosing one:

**Stay with `create_deep_agent`** for anything agentic — a model working a multi-step task with
tools. That is most agents, and the defaults encode real production experience you would
otherwise rediscover slowly.

**Drop to `create_agent`** when you want the agent loop but a materially different stack: your
own context strategy, no filesystem, a bespoke middleware ordering. You are still writing an
agent; you just want to choose the parts.

**Drop to raw LangGraph** when the thing you are building is **not an agent loop**:

- deterministic pipelines where a model is one step among many
- explicit branching a prompt should not be deciding (routing on a database value, say)
- multi-graph systems with their own control flow
- workflows where a human step sits *between* nodes rather than approving a tool call

The failure mode in both directions is real. Fighting the harness to get unusual control flow
means you wanted LangGraph. Hand-writing planning, file management, and summarization into a
raw graph means you wanted Deep Agents.

---

## 6. Where to go next

- **LangGraph docs** — state, nodes, edges, subgraphs, streaming, durability.
- **`create_agent`** — the middleware system in full, including hooks this course did not use.
- **These notebooks** — they are yours; the code runs, so change it and see what breaks.

You now have a complete picture of Part 1: an agent is a model in a loop, and everything
interesting is the harness around it — files, tools, delegation, seams, memory, procedures — all
running on a checkpointed graph.

**Part 2 asks a different question: how do you know it works?**

---

## 📌 Key takeaways

- `create_deep_agent` is `create_agent` plus a curated middleware stack — configuration, not a separate framework.
- Middleware appear for two reasons: you asked for them, or a capability argument installed them. Know which.
- Both constructors return a compiled **LangGraph** graph: state, nodes, edges, reducers.
- Reducers decide what a state update *means* — `messages` appends rather than overwrites.
- A checkpointer is one mechanism that gives you conversation memory, time travel, and durable interrupts.
- An interrupt is saved state, not a blocked process — which is why approval can happen the next morning.
- Drop to `create_agent` for a different stack; drop to LangGraph when it is not an agent loop at all.

---

## ➡️ Next

**[10 · Deploy it](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/10_deploy.ipynb)**

Everything so far has been judged by reading the output and deciding it looked right. Part 2
replaces that with a test suite — starting from a real failure in a real trace.